
# LSTM-based Exchange Rate Forecasting

This notebook demonstrates how to train and evaluate a PyTorch LSTM model to predict exchange rates from an Excel dataset.



## Configuration

Adjust the configuration dictionary below to match your dataset and experiment setup. You can choose which column is the dependent variable (target) and list the independent variables (features). Leave `feature_columns` as `None` to automatically use all columns other than the target as predictors.


In [ ]:

from copy import deepcopy
import math
from pathlib import Path
from typing import List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset

CONFIG = {
    'excel_path': Path('data/exchange_rates.xlsx'),  # Update to your file path
    'date_column': 'Date',
    'target_column': 'Close',  # Dependent variable
    'feature_columns': None,    # List[str] of independent variables or None to infer
    'sequence_length': 30,
    'test_size': 0.2,           # Fraction of samples reserved for testing
    'val_size': 0.1,            # Fraction of training samples reserved for validation
    'batch_size': 32,
    'epochs': 100,
    'learning_rate': 1e-3,
    'hidden_size': 64,
    'num_layers': 2,
    'dropout': 0.2,
    'patience': 10,             # Early stopping patience (set to 0 to disable)
    'min_delta': 1e-4,          # Minimum improvement to reset patience
    'seed': 42,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
CONFIG



## Load and Inspect Data


In [ ]:

df = pd.read_excel(CONFIG['excel_path'], parse_dates=[CONFIG['date_column']])
df = df.set_index(CONFIG['date_column']).sort_index()
print(f"Data shape: {df.shape}")
df.head()


In [ ]:
df.describe().T


## Prepare Features, Targets, and Sequences


In [ ]:

target = CONFIG['target_column']
if target not in df.columns:
    raise ValueError(f"Target column '{target}' not found in the dataset.")

if CONFIG['feature_columns'] is None:
    features = [col for col in df.columns if col != target]
else:
    features = CONFIG['feature_columns']

missing_cols = [col for col in features if col not in df.columns]
if missing_cols:
    raise ValueError(f"Feature columns not found in dataset: {missing_cols}")

feature_data = df[features].copy()
feature_data = feature_data.fillna(method='ffill').fillna(method='bfill')

target_series = df[target].copy().fillna(method='ffill').fillna(method='bfill')

scaler = StandardScaler()
scaled_features = scaler.fit_transform(feature_data.values)
processed_df = pd.DataFrame(scaled_features, index=feature_data.index, columns=features)
processed_df[target] = target_series.values
processed_df.head()


In [ ]:

def create_sequences(data: pd.DataFrame, feature_cols: List[str], target_col: str, seq_length: int):
    sequences = []
    targets = []
    for i in range(len(data) - seq_length):
        seq = data.iloc[i:i + seq_length][feature_cols].values
        label = data.iloc[i + seq_length][target_col]
        sequences.append(seq)
        targets.append(label)
    return np.array(sequences, dtype=np.float32), np.array(targets, dtype=np.float32)

seq_length = CONFIG['sequence_length']
if seq_length <= 0:
    raise ValueError('sequence_length must be a positive integer.')

X, y = create_sequences(processed_df, features, target, seq_length)
print(f"Sequences shape: {X.shape}")
print(f"Targets shape: {y.shape}")

# Random walk baseline uses the most recent actual value as the prediction
random_walk = target_series.values[seq_length - 1:-1].astype(np.float32)
print(f"Random walk baseline shape: {random_walk.shape}")



### Train/Validation/Test Split


In [ ]:

n_samples = len(X)
if n_samples == 0:
    raise ValueError('Not enough data points to create sequences. Reduce sequence_length or provide more data.')

# Determine split indices while ensuring each split has at least one sample
raw_test_size = max(1, int(np.floor(n_samples * CONFIG['test_size'])))
raw_test_size = min(raw_test_size, n_samples - 1)
train_end = n_samples - raw_test_size

raw_val_size = int(np.floor(train_end * CONFIG['val_size'])) if CONFIG['val_size'] > 0 else 0
if CONFIG['val_size'] > 0:
    raw_val_size = max(1, min(raw_val_size, train_end - 1))
val_start = train_end - raw_val_size if raw_val_size > 0 else train_end

X_train = X[:val_start]
y_train = y[:val_start]
X_val = X[val_start:train_end] if raw_val_size > 0 else np.empty((0, *X.shape[1:]))
y_val = y[val_start:train_end] if raw_val_size > 0 else np.empty((0,))
X_test = X[train_end:]
y_test = y[train_end:]

rw_train = random_walk[:val_start]
rw_val = random_walk[val_start:train_end] if raw_val_size > 0 else np.empty((0,))
rw_test = random_walk[train_end:]

index_after_sequences = processed_df.index[seq_length:]
train_dates = index_after_sequences[:val_start]
val_dates = index_after_sequences[val_start:train_end]
test_dates = index_after_sequences[train_end:]

print(f"Train samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Test samples: {len(X_test)}")



## Create Datasets and DataLoaders


In [ ]:

class SequenceDataset(Dataset):
    def __init__(self, sequences: np.ndarray, targets: np.ndarray):
        self.sequences = torch.from_numpy(sequences)
        self.targets = torch.from_numpy(targets)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]

train_dataset = SequenceDataset(X_train, y_train)
val_dataset = SequenceDataset(X_val, y_val) if len(X_val) > 0 else None
test_dataset = SequenceDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size']) if val_dataset is not None else None
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'])



## Define the LSTM Model


In [ ]:

class LSTMRegressor(nn.Module):
    def __init__(self, num_features: int, hidden_size: int, num_layers: int, dropout: float):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=num_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, _ = self.lstm(x)
        last_output = output[:, -1, :]
        return self.fc(last_output).squeeze(-1)

model = LSTMRegressor(
    num_features=len(features),
    hidden_size=CONFIG['hidden_size'],
    num_layers=CONFIG['num_layers'],
    dropout=CONFIG['dropout'],
).to(CONFIG['device'])

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])



## Train the Model with Validation-Based Model Selection


In [ ]:

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for sequences, targets in loader:
        sequences = sequences.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(sequences)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * sequences.size(0)
    return running_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    preds = []
    actuals = []
    with torch.no_grad():
        for sequences, targets in loader:
            sequences = sequences.to(device)
            targets = targets.to(device)
            outputs = model(sequences)
            loss = criterion(outputs, targets)
            running_loss += loss.item() * sequences.size(0)
            preds.extend(outputs.cpu().numpy())
            actuals.extend(targets.cpu().numpy())
    avg_loss = running_loss / len(loader.dataset)
    return avg_loss, np.array(preds), np.array(actuals)

history = {'train_loss': [], 'val_loss': []}
best_state = None
best_epoch = None
best_val_loss = math.inf
patience_counter = 0

for epoch in range(1, CONFIG['epochs'] + 1):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, CONFIG['device'])
    history['train_loss'].append(train_loss)

    if val_loader is not None and len(val_loader.dataset) > 0:
        val_loss, _, _ = evaluate(model, val_loader, criterion, CONFIG['device'])
        history['val_loss'].append(val_loss)

        if val_loss + CONFIG['min_delta'] < best_val_loss:
            best_val_loss = val_loss
            best_state = deepcopy(model.state_dict())
            best_epoch = epoch
            patience_counter = 0
        else:
            patience_counter += 1

        if CONFIG['patience'] > 0 and patience_counter >= CONFIG['patience']:
            print(f"Early stopping at epoch {epoch} (best epoch {best_epoch}).")
            break
    else:
        # If no validation set is provided, fall back to training loss for model selection
        if train_loss + CONFIG['min_delta'] < best_val_loss:
            best_val_loss = train_loss
            best_state = deepcopy(model.state_dict())
            best_epoch = epoch

    if epoch % 10 == 0 or epoch == 1:
        msg = f"Epoch {epoch:03d}: Train Loss = {train_loss:.6f}"
        if history['val_loss']:
            msg += f", Val Loss = {history['val_loss'][-1]:.6f}"
        print(msg)

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Loaded best model from epoch {best_epoch} with validation loss {best_val_loss:.6f}.")
else:
    print("No validation improvement recorded; using final model state.")


In [ ]:

plt.figure(figsize=(10, 4))
plt.plot(history['train_loss'], label='Train Loss')
if history['val_loss']:
    plt.plot(history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training History')
plt.legend()
plt.tight_layout()
plt.show()



## Evaluate on Test Set and Compare with Random Walk Baseline


In [ ]:

test_loss, test_preds, test_actuals = evaluate(model, test_loader, criterion, CONFIG['device'])
rmse = math.sqrt(mean_squared_error(test_actuals, test_preds))
mae = mean_absolute_error(test_actuals, test_preds)

rw_rmse = math.sqrt(mean_squared_error(test_actuals, rw_test))
rw_mae = mean_absolute_error(test_actuals, rw_test)

print(f"Test MSE: {test_loss:.6f}")
print(f"Test RMSE: {rmse:.6f}")
print(f"Test MAE: {mae:.6f}")
print("\nRandom Walk Baseline:")
print(f"Baseline RMSE: {rw_rmse:.6f}")
print(f"Baseline MAE: {rw_mae:.6f}")



## Plot Predictions vs Actuals


In [ ]:

plt.figure(figsize=(12, 5))
plt.plot(test_dates, test_actuals, label='Actual', linewidth=2)
plt.plot(test_dates, test_preds, label='LSTM Prediction', linestyle='--')
plt.plot(test_dates, rw_test, label='Random Walk', linestyle=':')
plt.xlabel('Date')
plt.ylabel(target)
plt.title('Actual vs Predicted Exchange Rates')
plt.legend()
plt.tight_layout()
plt.show()



## Save the Best Model (Optional)


In [ ]:

model_path = Path('models/lstm_exchange_rate.pth')
model_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    'model_state_dict': model.state_dict(),
    'scaler_mean': scaler.mean_,
    'scaler_scale': scaler.scale_,
    'config': CONFIG,
    'features': features,
    'target': target,
}, model_path)
print(f"Model saved to {model_path.resolve()}")
